<a href="https://colab.research.google.com/github/Jee8825/jeeva-codebooster-2026/blob/main/Day_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 744.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [2]:
print('Checking installation of chromadb:')
!pip show chromadb
print('\nChecking installation of sentence-transformers:')
!pip show sentence-transformers

Checking installation of chromadb:
Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 

Checking installation of sentence-transformers:
Name: sentence-transformers
Version: 5.5.1
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, num

In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb


In [4]:
documents = [
    "Sleep in the whisper of sirens.",
    "How long will they do ot die",
    "A majestic feline predator, the lion, roams the savanna.",
    "jeeva is a naughty boy.",
    "the pigeons are flying. ",
    "Automobiles are a common mode of transportation."
]

print("Our document corpus:")
for i, doc in enumerate(documents):
    print(f"[{i}] {doc}")

Our document corpus:
[0] Sleep in the whisper of sirens.
[1] How long will they do ot die
[2] A majestic feline predator, the lion, roams the savanna.
[3] jeeva is a naughty boy.
[4] the pigeons are flying. 
[5] Automobiles are a common mode of transportation.


In [5]:
def keyword_search(query, documents):
    results = []
    query_lower = query.lower()
    for i, doc in enumerate(documents):
        if query_lower in doc.lower():
            results.append((i, doc))
    return results

print("\n--- Keyword Search Results ---")
query_keyword = "dog"
keyword_results = keyword_search(query_keyword, documents)
print(f"Query: '{query_keyword}'")
if keyword_results:
    for idx, doc in keyword_results:
        print(f"Found in document [{idx}]: {doc}")
else:
    print("No exact keyword matches found.")

print("\n--- Keyword Search with synonym ---")
query_synonym = "canine"
keyword_results_synonym = keyword_search(query_synonym, documents)
print(f"Query: '{query_synonym}'")
if keyword_results_synonym:
    for idx, doc in keyword_results_synonym:
        print(f"Found in document [{idx}]: {doc}")
else:
    print("No exact keyword matches found.")


--- Keyword Search Results ---
Query: 'dog'
No exact keyword matches found.

--- Keyword Search with synonym ---
Query: 'canine'
No exact keyword matches found.


In [6]:
# Initialize the sentence-transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create a ChromaDB client and collection
import chromadb
client = chromadb.Client()
collection = client.get_or_create_collection(name="my_documents")

# Generate embeddings for the documents
document_embeddings = model.encode(documents).tolist()

# Add documents and their embeddings to ChromaDB
collection.add(
    embeddings=document_embeddings,
    documents=documents,
    metadatas=[{"source": f"doc_{i}"} for i in range(len(documents))],
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print("Embeddings generated and stored in ChromaDB.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generated and stored in ChromaDB.


In [7]:
def semantic_search(query, collection, model, n_results=2):
    # Generate embedding for the query
    query_embedding = model.encode([query]).tolist()

    # Query ChromaDB for similar documents
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    return results

print("\n--- Semantic Search Results ---")
query_semantic = "domestic pets"
semantic_results = semantic_search(query_semantic, collection, model, n_results=2)

print(f"Query: '{query_semantic}'")
if semantic_results and 'documents' in semantic_results and semantic_results['documents']:
    for i, doc_list in enumerate(semantic_results['documents']):
        for j, doc in enumerate(doc_list):
            distance = semantic_results['distances'][i][j]
            print(f"Relevant document (distance: {distance:.4f}): {doc}")
else:
    print("No semantic results found.")

print("\n--- Semantic Search with car synonym ---")
query_car_synonym = "vehicles for transport"
semantic_results_car = semantic_search(query_car_synonym, collection, model, n_results=1)

print(f"Query: '{query_car_synonym}'")
if semantic_results_car and 'documents' in semantic_results_car and semantic_results_car['documents']:
    for i, doc_list in enumerate(semantic_results_car['documents']):
        for j, doc in enumerate(doc_list):
            distance = semantic_results_car['distances'][i][j]
            print(f"Relevant document (distance: {distance:.4f}): {doc}")
else:
    print("No semantic results found.")


--- Semantic Search Results ---
Query: 'domestic pets'
Relevant document (distance: 1.1691): A majestic feline predator, the lion, roams the savanna.
Relevant document (distance: 1.6558): the pigeons are flying. 

--- Semantic Search with car synonym ---
Query: 'vehicles for transport'
Relevant document (distance: 0.6967): Automobiles are a common mode of transportation.


In [8]:
# Define a single sentence to embed
single_sentence = "This is a test sentence for embedding."

# Embed the sentence
sentence_embedding = model.encode([single_sentence])

# Print the shape of the embedding (to show it's a vector)
print(f"Original sentence: '{single_sentence}'")
print(f"Shape of the embedding: {sentence_embedding.shape}")

# Print the first few dimensions of the embedding (or the whole vector if small enough)
print(f"First 5 dimensions of the embedding: {sentence_embedding[0][:5]}...")

Original sentence: 'This is a test sentence for embedding.'
Shape of the embedding: (1, 384)
First 5 dimensions of the embedding: [0.02782412 0.00170262 0.08005547 0.04666286 0.03852206]...


In [9]:
# Perform a sample query using the first document as the query text
sample_query_text = five_docs[0]
query_results = collection.query(
    query_texts=[sample_query_text],
    n_results=1
)

# Display the keys of the result dictionary
print(f"Result keys: {query_results.keys()}")
print("\nFull result structure for reference:")
display(query_results)

NameError: name 'five_docs' is not defined

In [10]:
print(f"--- Search Results for: '{sample_query_text}' ---\n")

# Extract lists from the results dictionary
ids = query_results['ids'][0]
documents = query_results['documents'][0]
metadatas = query_results['metadatas'][0]
distances = query_results['distances'][0]

# Iterate and display in a readable format
for i in range(len(ids)):
    print(f"Match #{i+1}")
    print(f"ID: {ids[i]}")
    print(f"Text: {documents[i]}")
    print(f"Score (Distance): {distances[i]:.4f}")
    print(f"Metadata: {metadatas[i]}")
    print("-" * 30)

NameError: name 'sample_query_text' is not defined

In [ ]:
sample_docs = [
    "ETL stands for Extract, Transform and Load",
    "SQL SELECT statements retrieve data from database tables",
    "Machine learning models learn patterns from training data",
    "Python pandas library is used for data manipulation and cleaning",
    "Neural networks are inspired by how the human brain works",
]

sample_ids = ["doc001","doc002","doc003","doc004","doc005"]

sample_metadata = [
    {"subject": "Data engineering", "Topic": "ETL"},
    {"Subject": "Data engineering", "Topic": "SQL"},
    {"subject": "Machine learning", "Topic": "ML Basics"},
    {"subject": "Python", "Topic": "Pandas"},
    {"subject": "Machine learning", "Topic": "Neural networks"}
]

# Add the new documents to the existing collection
collection.add(
    documents=sample_docs,
    ids=sample_ids,
    metadatas=sample_metadata
)

print("Documents added to collection!")
print("Total Documents now in collection:", collection.count())

In [ ]:
filtered_query = "How do neural networks function?"

# Perform the filtered query
filtered_results = collection.query(
    query_texts=[filtered_query],
    n_results=3,
    where={"subject": "Machine learning"},
)

print("FILTERED QUERY:", filtered_query)
print("Filter: only machine learning documents")
print("="*60)

# Iterate through results using zip to handle multiple lists at once
for rank, (doc, dist, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
    filtered_results['metadatas'][0]
), start=1):
    # Handle inconsistent casing for the 'subject' key
    subject_val = meta.get('subject') or meta.get('Subject')
    topic_val = meta.get('Topic')

    print(f"Rank: {rank} | Distance: {dist:.4f} | subject: {subject_val} | Topic: {topic_val}")
    print(doc)
    print()

In [ ]:
search_query = "How do databases retrieve data using SQL?"

# Perform query
results = collection.query(
    query_texts=[search_query],
    n_results=3
)

print(f"--- Search Results for: '{search_query}' ---\n")

ids = results['ids'][0]
docs = results['documents'][0]
metas = results['metadatas'][0]
dists = results['distances'][0]

for i in range(len(ids)):
    print(f"Rank #{i+1} | ID: {ids[i]}")
    print(f"Text: {docs[i]}")
    print(f"Score (Distance): {dists[i]:.4f}")
    print(f"Metadata: {metas[i]}\n")

In [11]:
import chromadb

# 1. Prepare 15 notes with metadata
notes = [
    "Supervised learning uses labeled data.",
    "Unsupervised learning finds hidden patterns.",
    "Reinforcement learning is based on rewards.",
    "Deep learning uses neural networks.",
    "Python is great for data science.",
    "Pandas is used for data manipulation.",
    "Matplotlib helps in data visualization.",
    "SQL is essential for database queries.",
    "Natural Language Processing deals with text.",
    "Computer vision processes images.",
    "Overfitting occurs when a model is too complex.",
    "Gradient descent optimizes model weights.",
    "Random forests are ensembles of trees.",
    "Support Vector Machines find hyperplanes.",
    "Backpropagation updates neural net weights."
]

# Categorize subjects
subjects = ["Machine learning", "Machine learning", "Machine learning", "Machine learning", "General",
            "Data Tools", "Data Tools", "General", "Machine learning", "Machine learning",
            "Machine learning", "Machine learning", "Machine learning", "Machine learning", "Machine learning"]

ids = [f"note_{i}" for i in range(len(notes))]
metadatas = [{"subject": subjects[i], "id_tag": f"tag_{i}"} for i in range(len(notes))]

# 2. Setup ChromaDB
client = chromadb.Client()
collection = client.get_or_create_collection(name="smart_notes")

# Add to collection
collection.add(
    documents=notes,
    ids=ids,
    metadatas=metadatas
)

# 3. Define 5 different search queries
queries = [
    "What is neural network optimization?",
    "How to handle data in Python?",
    "Explain supervised learning.",
    "Image processing techniques.",
    "Ensemble methods in ML."
]

print("--- Search Results (Top 3 Distance Scores) ---\n")
for q in queries:
    results = collection.query(query_texts=[q], n_results=3)
    print(f"Query: {q}")
    for i in range(len(results['ids'][0])):
        print(f"  Match {i+1}: {results['documents'][0][i]} | Distance: {results['distances'][0][i]:.4f}")
    print("-" * 30)

# 4. Filtered search: Retrieve only Machine Learning notes
print("\n--- Filtered Results (Subject: Machine learning) ---")
ml_results = collection.query(
    query_texts=["General ML concepts"],
    n_results=5,
    where={"subject": "Machine learning"}
)

for i in range(len(ml_results['ids'][0])):
    print(f"Note: {ml_results['documents'][0][i]} | Metadata: {ml_results['metadatas'][0][i]}")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 10.4MiB/s]


--- Search Results (Top 3 Distance Scores) ---

Query: What is neural network optimization?
  Match 1: Gradient descent optimizes model weights. | Distance: 0.8948
  Match 2: Deep learning uses neural networks. | Distance: 0.9141
  Match 3: Backpropagation updates neural net weights. | Distance: 1.0416
------------------------------
Query: How to handle data in Python?
  Match 1: Python is great for data science. | Distance: 0.9309
  Match 2: Pandas is used for data manipulation. | Distance: 1.0972
  Match 3: Matplotlib helps in data visualization. | Distance: 1.1176
------------------------------
Query: Explain supervised learning.
  Match 1: Supervised learning uses labeled data. | Distance: 0.4371
  Match 2: Unsupervised learning finds hidden patterns. | Distance: 1.0157
  Match 3: Deep learning uses neural networks. | Distance: 1.0374
------------------------------
Query: Image processing techniques.
  Match 1: Computer vision processes images. | Distance: 0.6890
  Match 2: Natural